# Widespread Attack Detection (Late Stage)

## Overview

This notebook detects widespread attacks by identifying aggressive source IPs that target multiple hosts using WMI to control remote systems and exfiltrate data. The solution uses a sliding window approach combined with deep learning (Autoencoder) to identify anomalous process behavior on compromised systems.

**Objective:** Detect widespread attacks where multiple compromised devices are leveraged to control remote systems and encrypt business data.

**Data Requirements:**
- Sysmon Operational logs
- Essential Event IDs: 1 (Process Create), 3 (Network Connection), 11 (File Create), 19/20/21 (WMI)
- ProcessGuid field for sessionization
- Network activity (source/destination IPs)
- Data readable from MinIO/S3

**Expected Outputs:**
- Trained PyTorch Autoencoder model
- Blast radius map (aggressive source IPs and their targets)
- Correlated process/file activity on targeted hosts
- Anomaly scores for attack processes
- Campaign report (source IP → targets → anomalous actions)
- Confusion matrix and classification metrics

**Model Approach:**
**Phase 1 - Windowing:**
- Create sliding windows (10 min duration, 10 min slide)
- Cast timestamps to native Spark TimestampType
- Prepare for temporal aggregation

**Phase 2 - Blast Radius:**
- Identify aggressive source IPs (internal → internal connections)
- Count unique destinations per source IP per window
- Filter: IPs hitting ≥2 unique targets
- Build blast radius map (source → destination → hostname)

**Phase 3 - Correlation:**
- Join blast radius to process/file events on target hosts
- Correlate attack timing with victim activity
- Extract process lineage and file operations

**Phase 4 - Deep Learning:**
- Engineer TF-IDF features from command lines
- Train Autoencoder on normal baseline (5% sample)
- Score attack data with trained model
- High reconstruction error = anomalous (attack process)

**Reporting:**
- Group anomalies by source IP
- Filter campaigns (multiple windows, >3 targets)
- Human-readable report with timeline

**Prerequisites:**
- MinIO/S3 access configured in environment variables
- Spark cluster with GPU access
- Sysmon logs available
- PyTorch with GPU support

**Notes:**
- Developed in test environment; tuning is needed for your dataset
- Manual ground truth labeling required for metrics
- Threshold values (10.0, windows >1, targets >3) need adjustment
- IP-level metrics approach used (different from other notebooks)
- Can be adapted for other data sources (EDR solutions)

In [ ]:
# Cell 1: Imports & Environment Setup
# =========================================

import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, Bucketizer
from pyspark.ml.torch.distributor import TorchDistributor
import ipaddress
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.utils.data import IterableDataset, DataLoader
import s3fs
import json
from pyspark import TaskContext
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import traceback

load_dotenv()

print("✅ Environment loaded")

In [ ]:
# Cell 2: Spark Configuration
# =========================================

SPARK_APP_NAME = "Widespread_Attack_Detection"

SPARK_MASTER = os.getenv('SPARK_MASTER')
SPARK_DRIVER_HOST = os.getenv('SPARK_DRIVER_HOST')
SPARK_DRIVER_PORT = int(os.getenv('SPARK_DRIVER_PORT', '7771'))
SPARK_BLOCK_MANAGER_PORT = int(os.getenv('SPARK_BLOCK_MANAGER_PORT', '7772'))
SPARK_UI_PORT = int(os.getenv('SPARK_UI_PORT', '8088'))
SPARK_EXECUTOR_CORES = int(os.getenv('SPARK_EXECUTOR_CORES', '16'))
SPARK_EXECUTOR_MEMORY = os.getenv('SPARK_EXECUTOR_MEMORY', '32G')
SPARK_DRIVER_MEMORY = os.getenv('SPARK_DRIVER_MEMORY', '2g')
SPARK_EXECUTOR_GPU_AMOUNT = os.getenv('SPARK_EXECUTOR_GPU_AMOUNT', '1')
SPARK_TASK_GPU_AMOUNT = os.getenv('SPARK_TASK_GPU_AMOUNT', '1')
SPARK_JARS_PATH = os.getenv('SPARK_JARS_PATH', '/home/jmi/Spark_cluster/master/jars/*')
SPARK_EXECUTOR_CLASSPATH = os.getenv('SPARK_EXECUTOR_CLASSPATH', '/opt/spark/jars/*:/opt/spark/external-jars/*')
USE_GPU = os.getenv('USE_GPU', 'true').lower() == 'true'

print(f"✅ Spark Config: master={SPARK_MASTER}, cores={SPARK_EXECUTOR_CORES}, memory={SPARK_EXECUTOR_MEMORY}, gpu={USE_GPU}")

In [ ]:
# Cell 3: MinIO Configuration
# =========================================

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
MINIO_PATH_STYLE_ACCESS = os.getenv('MINIO_PATH_STYLE_ACCESS', 'true')

STORAGE_OPTIONS = {
    'key': MINIO_ACCESS_KEY,
    'secret': MINIO_SECRET_KEY,
    'endpoint_url': MINIO_ENDPOINT
}

print(f"✅ MinIO Config: endpoint={MINIO_ENDPOINT}")

In [ ]:
# Cell 4: Data Paths
# =========================================

DATA_PATH = os.getenv('DATA_PATH', 's3a://winlogbeat/winlogbeat/*.parquet')
BASE_S3_PATH = os.getenv('BASE_S3_PATH', 's3a://temp/widespread_attack')
BLAST_RADIUS_PATH = f"{BASE_S3_PATH}/blast_radius_map"
TRAIN_PATH = f"{BASE_S3_PATH}/train_parquet"
TEST_PATH = f"{BASE_S3_PATH}/test_parquet"

print(f"✅ Data Path: {DATA_PATH}")

In [ ]:
# Cell 5: Widespread Attack Detection Parameters
# =========================================

WS_WINDOW_DURATION = os.getenv('WS_WINDOW_DURATION', '60 minutes')
WS_WINDOW_SLIDE = os.getenv('WS_WINDOW_SLIDE', '10 minutes')
WS_RELEVANT_EVENTS = list(map(int, os.getenv('WS_RELEVANT_EVENTS', '1,3,11,19,20,21').split(',')))
WS_VICTIM_MIN_COUNT = int(os.getenv('WS_VICTIM_MIN_COUNT', '2'))
WS_TFIDF_NUM_FEATURES = int(os.getenv('WS_TFIDF_NUM_FEATURES', str(2**18)))
WS_AUTOENCODER_LATENT_DIM = int(os.getenv('WS_AUTOENCODER_LATENT_DIM', '64'))
WS_AUTOENCODER_LEARNING_RATE = float(os.getenv('WS_AUTOENCODER_LEARNING_RATE', '0.01'))
WS_AUTOENCODER_BATCH_SIZE = int(os.getenv('WS_AUTOENCODER_BATCH_SIZE', '64'))
WS_AUTOENCODER_MAX_BATCHES = int(os.getenv('WS_AUTOENCODER_MAX_BATCHES', '1000'))
WS_ANOMALY_THRESHOLD = float(os.getenv('WS_ANOMALY_THRESHOLD', '10.0'))
WS_NORMAL_SAMPLE_FRACTION = float(os.getenv('WS_NORMAL_SAMPLE_FRACTION', '0.05'))
# Anomaly reporting, how many windows & targets needed to be reported as an anomaly
WS_CAMPAIGN_MIN_WINDOWS = int(os.getenv('WS_CAMPAIGN_MIN_WINDOWS', '1'))
WS_CAMPAIGN_MIN_TARGETS = int(os.getenv('WS_CAMPAIGN_MIN_TARGETS', '3'))

print(f"✅ Widespread Attack Config: window={WS_WINDOW_DURATION}, victims_min={WS_VICTIM_MIN_COUNT}, threshold={WS_ANOMALY_THRESHOLD}")

In [ ]:
# Cell 6: Create Spark Session
# =========================================

builder = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .master(SPARK_MASTER)

    .config("spark.driver.host", SPARK_DRIVER_HOST)
    .config("spark.driver.port", str(SPARK_DRIVER_PORT))
    .config("spark.blockManager.port", str(SPARK_BLOCK_MANAGER_PORT))
    .config("spark.ui.port", str(SPARK_UI_PORT))

    .config("spark.executor.cores", str(SPARK_EXECUTOR_CORES))
    .config("spark.executor.memory", SPARK_EXECUTOR_MEMORY)
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)

    .config("spark.executor.extraClassPath", SPARK_EXECUTOR_CLASSPATH)
    .config("spark.jars", SPARK_JARS_PATH)
    .config("spark.executor.userClassPathFirst", "true")

    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", MINIO_PATH_STYLE_ACCESS)
)

if USE_GPU:
    builder = (
        builder
        .config("spark.executor.resource.gpu.amount", SPARK_EXECUTOR_GPU_AMOUNT)
        .config("spark.task.resource.gpu.amount", SPARK_TASK_GPU_AMOUNT)
    )

spark = builder.getOrCreate()

print(f"✅ Spark session created: {SPARK_APP_NAME}")
print(f"   Master: {SPARK_MASTER}")
print(f"   GPU: {USE_GPU}")

In [ ]:
# Cell 7: Load Data
# =========================================

df_raw = spark.read.parquet(DATA_PATH)
print(f"✅ Loaded {df_raw.count()} records from MinIO")

## Data Filtering

The following code:
1. Adds a human-readable time column from epoch time
2. Filters data for relevant Event IDs (1, 3, 11, 19, 20, 21)
3. Removes unnecessary columns to reduce memory footprint
4. Filters for non-null ProcessGuid

**Key Fields:**
- `event_created`: Event creation time in epoch
- `winlog_event_id`: Sysmon event ID
- `winlog_event_data_processguid`: Unique process session identifier
- `winlog_event_data_sourceip`: Source IP address
- `winlog_event_data_destinationip`: Destination IP address
- `winlog_computer_name`: Hostname

In [ ]:
# Cell 8: Filter Data
# =========================================

df_with_time = df_raw.withColumn(
    "readable_time",
    F.from_unixtime(F.col("event_created") / 1000000000)
)

CleanData = (
    df_with_time
    .where(F.col("winlog_channel") == "Microsoft-Windows-Sysmon/Operational")
    .where(F.col("winlog_event_id").isin(WS_RELEVANT_EVENTS))
    .where(F.col("winlog_event_data_processguid").isNotNull())
    .drop(
        "_index", "_id", "_timestamp", "@timestamp", "@version",
        "tags", "message", "ecs_version", "event_kind", "event_type",
        "event_module", "event_dataset", "agent_ephemeral_id", "agent_id",
        "agent_name", "agent_type", "agent_version",
        "winlog_keywords", "winlog_activity_id", "winlog_provider_guid",
        "winlog_event_data_utctime", "winlog_event_data_creationutctime",
        "winlog_user_identifier", "log_level", "event_action",
        "winlog_event_data_configurationfilehash", "winlog_event_data_configuration",
        "winlog_event_data_elevatedtoken", "winlog_event_data_workstationname",
        "winlog_event_data_ipport", "winlog_event_data_targetdomainname",
        "winlog_event_data_authenticationpackagename", "winlog_event_data_targetusername",
        "winlog_event_data_subjectdomainname", "winlog_event_data_targetusersid",
        "winlog_event_data_subjectusersid", "winlog_event_data_restrictedadminmode",
        "winlog_event_data_impersonationlevel", "winlog_event_data_targetlogonid",
        "winlog_event_data_transmittedservices", "winlog_event_data_subjectlogonid",
        "winlog_event_data_virtualaccount", "winlog_event_data_subjectusername",
        "winlog_event_data_targetoutboundusername", "winlog_event_data_scriptblockid",
        "winlog_event_data_targetoutbounddomainname", "winlog_event_data_lmpackagename",
        "winlog_event_data_targetlinkedlogonid", "winlog_event_data_service",
        "winlog_event_data_objectserver", "winlog_event_data_privilegelist",
        "winlog_event_data_accessmask", "winlog_event_data_objecttype",
        "winlog_event_data_objectname", "winlog_event_data_accesslist",
        "winlog_event_data_calltrace",
        "winlog_event_data_resourceattributes", "winlog_event_data_keylength",
        "winlog_event_data_messagenumber", "winlog_event_data_messagetotal",
        "winlog_event_data_schemaversion", "winlog_event_data_version"
    )
)

print(f"✅ Filtered to {CleanData.count()} records (Sysmon Ops, event IDs {WS_RELEVANT_EVENTS}, non-null processguid)")

## Phase 1: Windowing

Implement sliding window logic to identify widespread attacks within timeframes.

**Process:**
1. Cast readable_time to native TimestampType
2. Apply sliding windows (10 min duration, 10 min slide)
3. Prepare for temporal aggregation per window

**Note:** Sliding windows allow detecting attacks that span multiple time periods

In [ ]:
# Cell 9: Apply Sliding Windows
# =========================================

print("Starting Phase: Windowing...")

df_time_stamped = CleanData.withColumn(
    "event_ts",
    F.to_timestamp("readable_time")
)

df_windowed = df_time_stamped.withColumn(
    "window_spec",
    F.window("event_ts", WS_WINDOW_DURATION, WS_WINDOW_SLIDE)
)

print(f"✅ Windowing complete. {df_windowed.count()} records in {WS_WINDOW_DURATION} windows")

## Phase 2: Blast Radius Analysis

Identify aggressive source IPs that target multiple internal hosts.

**Process:**
1. Filter for network events (Event ID 3)
2. Define UDF for internal IP detection
3. Filter for internal-to-internal connections only
4. Group by window and source IP
5. Count unique destinations per source
6. Filter: IPs hitting ≥2 unique targets
7. Build blast radius map (source → destination → hostname)

**Result:** List of aggressive IPs and their target hosts per window

In [ ]:
# Cell 10: Blast Radius Analysis
# =========================================

print("Starting Phase: Blast Radius Analysis...")

def is_internal(ip_str):
    if not ip_str:
        return False
    try:
        return ipaddress.ip_address(ip_str).is_private
    except:
        return False

is_internal_udf = F.udf(is_internal, T.BooleanType())

network_events = (
    df_windowed
    .filter(F.col("winlog_event_id") == 3)
    .filter(
        (is_internal_udf("winlog_event_data_sourceip") == True) &
        (is_internal_udf("winlog_event_data_destinationip") == True)
    )
)

aggressors = (
    network_events
    .groupBy("window_spec", "winlog_event_data_sourceip")
    .agg(
        F.countDistinct("winlog_event_data_destinationip").alias("Victim_Count"),
        F.collect_set("winlog_event_data_destinationhostname").alias("Target_Hostnames_List")
    )
    .filter(F.col("Victim_Count") >= WS_VICTIM_MIN_COUNT)
)

blast_radius_df = (
    network_events.join(
        F.broadcast(aggressors),
        ["window_spec", "winlog_event_data_sourceip"],
        "inner"
    )
    .select(
        "window_spec",
        "winlog_event_data_sourceip",
        "winlog_event_data_destinationip",
        "winlog_event_data_destinationhostname"
    )
    .distinct()
)

print(f"✅ Blast radius complete. {blast_radius_df.count()} relationships detected")

blast_radius_df.write.mode("overwrite").parquet(BLAST_RADIUS_PATH)
print(f"✅ Blast radius saved to {BLAST_RADIUS_PATH}")

## Phase 3: Correlation

Correlate blast radius with process/file activity on targeted hosts.

**Process:**
1. Load blast radius map from previous phase
2. Filter for process and file creation events (Event IDs 1, 11)
3. Join blast radius to victim activity (match window and hostname)
4. Extract process lineage, command lines, and file operations

**Result:** Process/file events on hosts being actively attacked

In [ ]:
# Cell 11: Correlate Attack Activity
# =========================================

print("Starting Phase: Correlation...")

blast_radius = spark.read.parquet(BLAST_RADIUS_PATH)

victim_activity_raw = df_windowed.filter(F.col("winlog_event_id").isin([1, 11]))

v = victim_activity_raw.alias("v")
b = blast_radius.alias("b")

correlated_activity = v.join(
    F.broadcast(b),
    (
        v["window_spec"] == b["window_spec"]
    ) & (
        v["winlog_computer_name"].contains(b["winlog_event_data_destinationhostname"])
    ),
    "inner"
)

final_dataset = correlated_activity.select(
    v["window_spec"],
    v["winlog_computer_name"],
    v["winlog_event_id"],
    b["winlog_event_data_sourceip"],
    v["winlog_event_data_processguid"],
    v["winlog_event_data_image"],
    v["winlog_event_data_commandline"],
    v["winlog_event_data_parentimage"],
    v["winlog_event_data_parentcommandline"],
    v["winlog_event_data_targetfilename"]
)

print(f"✅ Correlation complete. {final_dataset.count()} artifacts correlated")

## Phase 4: Deep Learning & Anomaly Detection

Train Autoencoder to detect anomalous attack processes.

**Architecture:**
- Input: TF-IDF features from command lines
- Encoder: Compresses to latent space (64 dimensions)
- Decoder: Reconstructs input from latent space
- Training: Minimize reconstruction error (MSE)
- Inference: High reconstruction error = anomalous

**Hyperparameters:**
- `TFIDF_NUM_FEATURES`: Hashing TF features (2^18)
- `LATENT_DIM`: Latent space dimension (64)
- `LEARNING_RATE`: 0.01
- `BATCH_SIZE`: 64
- `MAX_BATCHES`: 1000 (early stopping)
- `ANOMALY_THRESHOLD`: 10.0 (reconstruction error cutoff)

**Distributed Training:**
- Use TorchDistributor for GPU acceleration
- Workers read training data from MinIO/S3
- Each worker loads PyTorch model on GPU
- Reduces training time significantly

In [ ]:
# Cell 12: Prepare Training & Test Data
# =========================================

print("Starting Phase: Prepare Deep Learning Data...")

def prep_data(df):
    return df.withColumn(
        "text_to_tokenize",
        F.concat_ws(
            " ",
            F.coalesce(F.col("winlog_event_data_parentcommandline"), F.lit("")),
            F.coalesce(F.col("winlog_event_data_commandline"), F.lit("")),
            F.coalesce(F.col("winlog_event_data_targetfilename"), F.lit(""))
        )
    ).withColumn("window_start", F.col("window_spec.start"))

normal_data_raw = df_windowed.filter(F.col("winlog_event_id").isin([1, 11])).sample(False, WS_NORMAL_SAMPLE_FRACTION)
normal_data = prep_data(normal_data_raw)
attack_data = prep_data(final_dataset)

tokenizer = Tokenizer(inputCol="text_to_tokenize", outputCol="cmd_tokens")
hashingTF = HashingTF(inputCol="cmd_tokens", outputCol="cmd_raw_features", numFeatures=WS_TFIDF_NUM_FEATURES)
idf = IDF(inputCol="cmd_raw_features", outputCol="cmd_tfidf_features")

pipeline = Pipeline(stages=[tokenizer, hashingTF, idf])
pipeline_model = pipeline.fit(normal_data)

normal_featured = pipeline_model.transform(normal_data).select(
    "text_to_tokenize", "cmd_tfidf_features"
)
attack_featured = pipeline_model.transform(attack_data).select(
    "winlog_event_data_sourceip", "window_start", "winlog_computer_name",
    "text_to_tokenize", "cmd_tfidf_features"
)

print(f"Writing Training Data to {TRAIN_PATH}...")
normal_featured.write.mode("overwrite").parquet(TRAIN_PATH)

print(f"Writing Attack Data to {TEST_PATH}...")
attack_featured.write.mode("overwrite").parquet(TEST_PATH)

print(f"✅ Data preparation complete")

In [ ]:
# Cell 13: Autoencoder Training Function
# =========================================

class BAutoencoder(nn.Module):
    def __init__(self, sparse_dim, latent_dim=64):
        super().__init__()
        self.sparse_embed = nn.EmbeddingBag(sparse_dim, 64, mode='sum')
        input_dim = 64
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, input_dim))

    def forward(self, indices, values, offsets):
        x = self.sparse_embed(indices, per_sample_weights=values, offsets=offsets)
        z = self.encoder(x)
        return z, self.decoder(x)

class S3ParquetIterableDataset(IterableDataset):
    def __init__(self, s3_path, storage_options, is_train=True):
        self.fs = s3fs.S3FileSystem(**storage_options)
        self.s3_path = s3_path
        self.is_train = is_train

    def __iter__(self):
        files = self.fs.glob(self.s3_path + "/*.parquet")
        
        for f in files:
            dataset = pq.ParquetDataset(f, filesystem=self.fs)
            table = dataset.read()
            df = table.to_pandas()
            
            for _, row in df.iterrows():
                sv = row['cmd_tfidf_features']
                
                if isinstance(sv, dict):
                    indices = torch.tensor(sv['indices'], dtype=torch.long)
                    values = torch.tensor(sv['values'], dtype=torch.float)
                else:
                    indices = torch.tensor(sv.indices.tolist(), dtype=torch.long)
                    values = torch.tensor(sv.values.tolist(), dtype=torch.float)

                offsets = torch.tensor([0], dtype=torch.long)
                
                data = {
                    'indices': indices,
                    'values': values,
                    'offsets': offsets
                }
                
                if self.is_train:
                    yield data
                else:
                    yield {
                        **data,
                        'source_ip': row['winlog_event_data_sourceip'],
                        'window_start': row['window_start'],
                        'hostname': row['winlog_computer_name'],
                        'text': row['text_to_tokenize']
                    }

def collate_fn(batch):
    indices = [x['indices'] for x in batch]
    values = [x['values'] for x in batch]
    offsets = [x['offsets'] for x in batch]
    
    flat_indices = torch.cat(indices)
    flat_values = torch.cat(values)
    
    lens = [len(t) for t in indices]
    batch_offsets = torch.cat([torch.tensor([0]), torch.cumsum(torch.tensor(lens[:-1]), dim=0)])
    
    return flat_indices, flat_values, batch_offsets

def detect_anomalies_fn(kwargs):
    """
    Train Autoencoder and detect anomalies using TorchDistributor.

    Args:
        kwargs: Dict with train_path, test_path, storage_options

    Returns:
        List of anomaly dictionaries
    """
    try:
        dist.init_process_group(backend="nccl")
        device = torch.device("cuda")
        print(f"[WORKER] Training on device: {device}")

        train_path = kwargs['train_path']
        test_path = kwargs['test_path']
        storage_options = kwargs['storage_options']
        
        train_ds = S3ParquetIterableDataset(train_path, storage_options, is_train=True)
        test_ds = S3ParquetIterableDataset(test_path, storage_options, is_train=False)
        
        train_loader = DataLoader(train_ds, batch_size=WS_AUTOENCODER_BATCH_SIZE, collate_fn=collate_fn)
        
        model = BAutoencoder(sparse_dim=WS_TFIDF_NUM_FEATURES, latent_dim=WS_AUTOENCODER_LATENT_DIM).to(device)
        optimizer = optim.Adam(model.parameters(), lr=WS_AUTOENCODER_LEARNING_RATE)
        criterion = nn.MSELoss()
        
        model.train()
        epoch_cnt = 0
        for batch in train_loader:
            idx, val, offs = batch
            idx, val, offs = idx.to(device), val.to(device), offs.to(device)
            
            z, recon = model(idx, val, offs)
            loss = criterion(recon, z)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_cnt += 1
            if epoch_cnt > WS_AUTOENCODER_MAX_BATCHES:
                break
            
        print(f"[WORKER] Training complete: {epoch_cnt} batches")
        
        model.eval()
        anomalies = []
        
        with torch.no_grad():
            for row in test_ds:
                indices = row['indices'].to(device)
                values = row['values'].to(device)
                offsets = row['offsets'].to(device)
                
                z, recon = model(indices, values, offsets)
                error = criterion(recon, z).item()
                
                if error > WS_ANOMALY_THRESHOLD:
                    anomalies.append({
                        'source_ip': row['source_ip'],
                        'window_start': row['window_start'],
                        'hostname': row['hostname'],
                        'text': row['text'],
                        'error': error
                    })

        dist.destroy_process_group()
        return anomalies
        
    except Exception as e:
        print("[WORKER] ERROR:")
        traceback.print_exc()
        raise e

print("✅ Training function defined")

In [ ]:
# Cell 14: Launch Distributed Training & Inference
# =========================================

print("Starting Phase: Distributed Training & Inference on GPU...")

kwargs = {
    'train_path': TRAIN_PATH,
    'test_path': TEST_PATH,
    'storage_options': STORAGE_OPTIONS
}

distributor = TorchDistributor(num_processes=1, local_mode=False, use_gpu=USE_GPU)

print("Running Training & Inference...")
results = distributor.run(detect_anomalies_fn, kwargs)

print(f"✅ Phase: Distributed Training & Inference complete")

## Campaign Reporting

Aggregate anomalies and generate human-readable campaign reports.

**Process:**
1. Convert results to pandas DataFrame
2. Group by source IP
3. Aggregate campaign statistics (time windows, targets, actions)
4. Filter for campaigns:
   - Multiple time windows (>1)
   - Multiple targets (>3)
5. Print formatted report

**Result:** List of attack campaigns with timeline and impact

In [ ]:
# Cell 15: Generate Campaign Report
# =========================================

# Change the default campaing variables if needed
WS_CAMPAIGN_MIN_WINDOWS = 3
WS_CAMPAIGN_MIN_TARGETS = 2

print("Starting Phase: Campaign Reporting...")

df_res = pd.DataFrame(results)

if not df_res.empty:
    df_res = df_res.sort_values(by='window_start')
    
    final_report = df_res.groupby(['source_ip']).agg(
        Start_Time=('window_start', 'min'),
        End_Time=('window_start', 'max'),
        Windows_Involved=('window_start', 'nunique'),
        Target_Hosts=('hostname', lambda x: list(dict.fromkeys(x))),
        Target_Hosts_Count=('hostname', lambda x: len(list(dict.fromkeys(x)))),
        Anomalous_Actions=('text', lambda x: list(dict.fromkeys(x))),
        Total_Anomalies=('text', 'count')
    ).reset_index()
    
    print("Displaying the final report to show the high level information of the potential campaigns.")
    print("The WS_CAMPAIGN_MIN_WINDOWS and WS_CAMPAIGN_MIN_TARGETS variables can be adjusted accordingly")
    display(final_report)

    final_report_filtered = final_report[
        (final_report['Windows_Involved'] > WS_CAMPAIGN_MIN_WINDOWS) &
        (final_report['Target_Hosts_Count'] > WS_CAMPAIGN_MIN_TARGETS)
    ]
    final_report_filtered = final_report_filtered.sort_values(by='Windows_Involved', ascending=False)
    
    print("\n" + "=" * 100)
    print("BLAST RADIUS ATTACK CAMPAIGNS")
    print("=" * 100)
    
    for index, row in final_report_filtered.iterrows():
        print(f"\nSource IP        : {row['source_ip']}")
        print(f"Time Window      : {row['Start_Time']} to {row['End_Time']}")
        print(f"Windows Affected  : {row['Windows_Involved']}")
        print(f"Total Anomalies  : {row['Total_Anomalies']}")
        print(f"Target host count  : {row['Target_Hosts_Count']}")
        
        print(f"\nTarget Hosts ({len(row['Target_Hosts'])}):")
        for host in row['Target_Hosts']:
            print(f"  - {host}")
        
        print(f"\nAnomalous Actions ({len(row['Anomalous_Actions'])}):")
        for action in row['Anomalous_Actions']:
            print(f"  - {action}")
        
        print("-" * 100)
    
    final_report_final = final_report_filtered
else:
    print("No anomalies found.")
    final_report_final = pd.DataFrame()

print(f"✅ Campaign reporting complete")

In [ ]:
# Cell 16: Metrics Helper Function
# =========================================

def calculate_metrics(tp, fp, fn, tn, total_events=None):
    """
    Calculate classification metrics with standard output format.

    Args:
        tp: True Positives
        fp: False Positives
        fn: False Negatives
        tn: True Negatives
        total_events: Optional total for report

    Returns:
        dict with all metrics or None if error
    """
    try:
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        print("\n" + "=" * 40)
        print("CLASSIFICATION METRICS REPORT")
        print("=" * 40)
        if total_events:
            print(f"Total Events Evaluated: {total_events}")
        print(f"True Positives (TP):  {tp:,.0f}")
        print(f"False Positives (FP): {fp:,.0f}")
        print(f"False Negatives (FN): {fn:,.0f}")
        print(f"True Negatives (TN):  {tn:,.0f}")
        print("-" * 40)
        print(f"Accuracy:      {accuracy:.4f}")
        print(f"Precision:     {precision:.4f}")
        print(f"Recall:        {recall:.4f}")
        print(f"F1-Measure:    {f_measure:.4f}")
        print(f"False Pos Rate: {fpr:.4f}")
        print(f"False Neg Rate: {fnr:.4f}")
        print("=" * 40)

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f_measure': f_measure,
            'fpr': fpr,
            'fnr': fnr
        }
    except ZeroDivisionError as e:
        print(f"❌ ERROR: Division by zero in metrics calculation: {e}")
        return None

print("✅ Metrics helper function defined")

In [ ]:
# Cell 17: Visualization Helper Function
# =========================================

def plot_confusion_matrix(tp, fp, fn, tn, title='Confusion Matrix'):
    """
    Plot confusion matrix with consistent styling across all notebooks.

    Args:
        tp, fp, fn, tn: Confusion matrix values
        title: Chart title
    """
    matrix_values = [[tp, fn], [fp, tn]]

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        matrix_values,
        annot=True,
        fmt=',.0f',
        cmap='Purples',
        linewidths=2,
        linecolor='white',
        xticklabels=['Predicted Positive', 'Predicted Negative'],
        yticklabels=['Actual Positive', 'Actual Negative'],
        cbar=False,
        annot_kws={"size": 16, "weight": "bold"}
    )

    plt.title(title, fontsize=18, fontweight='bold', pad=20)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('Actual Label', fontsize=14)
    plt.tick_params(labelsize=12)
    plt.tight_layout()
    plt.show()

print("✅ Visualization helper function defined")

In [ ]:
# Cell 18: Calculate Confusion Matrix & Metrics
# =========================================

ground_truth_labels = {
    '10.0.3.154': 'MALICIOUS'
}

ref_df = (
    blast_radius_df
    .select("winlog_event_data_sourceip")
    .distinct()
    .toPandas()
)
all_ips = ref_df['winlog_event_data_sourceip'].tolist()

if not final_report_final.empty:
    predicted_ips = final_report_final['source_ip'].tolist()
    total_events = len(all_ips)
else:
    predicted_ips = []
    total_events = len(all_ips)

print(f"Total Events Analyzed (N): {total_events}")
print(f"Total Distinct Source IPs: {len(all_ips)}")
print(f"Detected Anomalies (Campaigns): {len(predicted_ips)}")

tp = 0
fp = 0
fn = 0
tn = 0

for ip in all_ips:
    label = ground_truth_labels.get(ip)
    if label is None:
        continue

    if ip in predicted_ips:
        if label == 'MALICIOUS':
            tp += 1
        else:
            fp += 1
    else:
        if label == 'MALICIOUS':
            fn += 1
        else:
            tn += 1

tn = total_events - (tp + fp + fn)

print(f"\nConfusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")

metrics = calculate_metrics(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    total_events=total_events
)

In [ ]:
# Cell 19: Plot Confusion Matrix
# =========================================

plot_confusion_matrix(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    title='Confusion Matrix - Widespread Attack Detection'
)

## Conclusion

This notebook successfully:
- Configured Spark with environment variables (including GPU support)
- Loaded and filtered Sysmon logs (Event IDs 1, 3, 11, 19, 20, 21)
- Applied sliding windows for temporal analysis
- Identified aggressive source IPs via blast radius analysis
- Correlated attack timing with victim process/file activity
- Trained PyTorch Autoencoder for anomaly detection
- Generated campaign reports with timeline and impact
- Calculated IP-level classification metrics

**Key Insights:**
- Sliding windows enable detection of multi-phase attacks
- Blast radius identifies IPs targeting multiple internal hosts
- TF-IDF captures process lineage patterns effectively
- Campaign grouping reveals coordinated attack patterns
- Manual ground truth labeling required for metrics
- IP-level metrics approach differs from other notebooks

**Next Steps:**
- Review campaign reports for suspicious patterns
- Investigate high-impact campaigns (many targets/windows)
- Check anomalous process actions on targeted hosts
- Tune hyperparameters (window size, TF-IDF features, thresholds)
- Update ground truth labels for your threat landscape
- Consider implementing automated alerts for new campaigns

**Configuration Changes:**
- Modify `.env` file or set environment variables for:
  - Spark cluster settings (master, GPU flags, memory, cores)
  - MinIO credentials and endpoint
  - Data paths
  - Model hyperparameters:
    - Windowing: duration, slide
    - TF-IDF: num_features
    - Autoencoder: latent_dim, learning_rate, batch_size, max_batches
  - Detection thresholds:
    - Anomaly threshold (reconstruction error)
    - Campaign filters (min windows, min targets)
  - Sampling fraction for normal baseline

**Notes:**
- Manual ground truth labeling required for metrics
- Threshold values (10.0, windows >1, targets >3) need adjustment
- IP-level metrics approach used (different from other notebooks)
- Ground truth labels must be updated for your environment
- Campaign filters identify coordinated multi-window attacks